### Building the final dataset

This notebook builds the complete dataset for the TV shows project, as no single dataset is available online. We will combine three main sources:

- **IMDb Datasets:** Episode ratings to calculate Season 1 drop-off and slope.  
  Source: [IMDb](https://datasets.imdbws.com/)

- **TMDB Dataset:** Target variable (cancellation status) and network features, last updated in 2024.  
  Source: [Kaggle](https://www.kaggle.com/)

- **TVMaze API:** Episode air dates to engineer the release strategy feature: **Binge** (all episodes released together) vs. **Weekly** (episodes released weekly).  
  Source: [TVMaze](https://www.tvmaze.com/api)

In [1]:
# Libraries and modules
import pandas as pd
import numpy as np

In [2]:
# First dataset
path = "/home/fra/TV_SHOW_DATASET/"
file_name = path + "title.basics.tsv.gz"
basics = pd.read_csv(file_name, sep="\t", na_values="\\N", low_memory=False)

In [3]:
print(basics.shape)
basics.head()

(12866375, 9)


,tconst,titleType,primaryTitle,originalTitle,isAdult,startYear,endYear,runtimeMinutes,genres
0,tt0000001,short,Carmencita,Carmencita,0,1894.0,NaN,1,"Documentary,Short"
1,tt0000002,short,Le clown et ses chiens,Le clown et ses chiens,0,1892.0,NaN,5,"Animation,Short"
2,tt0000003,short,Poor Pierrot,Pauvre Pierrot,0,1892.0,NaN,5,"Animation,Comedy,Romance"
3,tt0000004,short,Un bon bock,Un bon bock,0,1892.0,NaN,12,"Animation,Short"
4,tt0000005,short,Blacksmith Scene,Blacksmith Scene,0,1893.0,NaN,1,Short


In [4]:
basics.info()

<class 'pandas.DataFrame'>
RangeIndex: 12866375 entries, 0 to 12866374
Data columns (total 9 columns):
 #   Column          Dtype  
---  ------          -----  
 0   tconst          str    
 1   titleType       str    
 2   primaryTitle    str    
 3   originalTitle   str    
 4   isAdult         int64  
 5   startYear       float64
 6   endYear         float64
 7   runtimeMinutes  str    
 8   genres          str    
dtypes: float64(2), int64(1), str(6)
memory usage: 883.5 MB


In [5]:
basics.isna().sum()

tconst                   0
titleType                0
primaryTitle            25
originalTitle           25
isAdult                  0
startYear          1485546
endYear           12703400
runtimeMinutes     8009786
genres              544915
dtype: int64

In [6]:
basics["titleType"].value_counts()

titleType
tvEpisode       9950953
short           1163505
movie            759560
video            332285
tvSeries         306994
tvMovie          156595
tvMiniSeries      73606
tvSpecial         61158
videoGame         50637
tvShort           11081
tvPilot               1
Name: count, dtype: int64

### We focus only on TV shows released after 2015 (The streaming platforms era)


In [22]:
tv_series = basics[(basics["titleType"] == "tvSeries") & 
    (basics["startYear"] >= 2015)].copy()

In [23]:
tv_series.head()

,tconst,titleType,primaryTitle,originalTitle,isAdult,startYear,endYear,runtimeMinutes,genres
90921,tt0092999,tvSeries,Eyes on the Prize III: We Who Believe in Freed...,Eyes on the Prize III: We Who Believe in Freed...,0,2025.0,2025.0,60,"Documentary,History"
111124,tt0113735,tvSeries,The Magic 7,The Magic 7,0,2026.0,NaN,NaN,"Adventure,Animation,Family"
153394,tt0158429,tvSeries,Sauda,Sauda,0,2018.0,NaN,NaN,NaN
161201,tt0166938,tvSeries,Yo-TV,Yo-TV,0,2020.0,NaN,NaN,NaN
181274,tt0188360,tvSeries,Show of Hearts,Show of Hearts,0,2015.0,NaN,1320,Comedy


### Uploading the dataset containing all the episodes


In [27]:
file_name = path + "title.episode.tsv.gz"
episodes = pd.read_csv(file_name, sep = "\t", na_values="\\N")

In [28]:
episodes.head()

,tconst,parentTconst,seasonNumber,episodeNumber
0,tt0031458,tt32857063,NaN,NaN
1,tt0041951,tt0041038,1.0,9.0
2,tt0042816,tt0989125,1.0,17.0
3,tt0042889,tt0989125,NaN,NaN
4,tt0043426,tt0040051,3.0,42.0


In [29]:
episodes.dtypes

tconst               str
parentTconst         str
seasonNumber     float64
episodeNumber    float64
dtype: object

In [30]:
s1_eps = episodes[(episodes["seasonNumber"] == 1) & 
    (episodes["parentTconst"].isin(tv_series["tconst"]))].dropna(subset = ["episodeNumber"])

In [32]:
# upload ratings
file_name = path + "title.ratings.tsv.gz"
ratings = pd.read_csv(file_name, sep = "\t", na_values="\\N")

In [33]:
ratings.head()

,tconst,averageRating,numVotes
0,tt0000001,5.7,2234
1,tt0000002,5.4,327
2,tt0000003,6.4,2395
3,tt0000004,5.0,203
4,tt0000005,6.2,3100


In [40]:
# Merge ratings to episodes of season 1
s1_rated = s1_eps.merge(ratings, on="tconst", how = "inner")
s1_rated.sort_values(["parentTconst","episodeNumber"])
s1_rated.head()

,tconst,parentTconst,seasonNumber,episodeNumber,averageRating,numVotes
0,tt0937104,tt6264782,1.0,20.0,5.2,17
1,tt10002038,tt9784004,1.0,5.0,8.8,7
2,tt10002430,tt9606818,1.0,4.0,5.9,25
3,tt10002516,tt9475692,1.0,10.0,6.5,26
4,tt10003090,tt7762780,1.0,10.0,7.0,8


In [42]:
# Feature Engineering
def calc_s1_features(df):
    if len(df) < 3 or df["numVotes"].iloc[0] < 50:
        # remove series with fewer than 3 rated episodes or pilot episodes with fewer than 50 votes
        return None
        
    votes = df["numVotes"].values
    rats = df["averageRating"].values
    ep_nums = np.arange(len(rats))
    
    slope = np.polyfit(ep_nums, rats, 1)[0]  # Rating slope in Season 1
    drop_off = votes[-1] / votes[0]          # Retention: final episode votes / pilot votes
    
    return pd.Series({
        "s1_ep_count": len(df),
        "s1_mean_rating": np.mean(rats),
        "s1_std_rating": np.std(rats),
        "s1_rating_slope": slope,
        "s1_vote_dropoff": drop_off,
        "pilot_votes": votes[0]
    })

In [44]:
# Compress episodes in a single row per series
s1_features = s1_rated.groupby("parentTconst").apply(calc_s1_features).dropna().reset_index()
s1_features.head()

,parentTconst,s1_ep_count,s1_mean_rating,s1_std_rating,s1_rating_slope,s1_vote_dropoff,pilot_votes
0,tt0429087,12.0,7.700000,0.173205,-0.005594,0.608033,722.0
1,tt0475784,10.0,8.860000,0.438634,0.112727,1.340722,25129.0
2,tt0489974,8.0,7.825000,0.326917,0.114286,0.809496,3454.0
3,tt0804484,10.0,7.490000,0.403609,0.049091,0.543780,7709.0
4,tt0808224,6.0,7.333333,0.418994,0.120000,0.685185,162.0


In [45]:
# Merge with TV shows
imdb_df = tv_series.merge(s1_features, left_on="tconst", right_on="parentTconst").drop(columns=["parentTconst"])
imdb_df.head(10)

,tconst,titleType,primaryTitle,originalTitle,isAdult,startYear,endYear,runtimeMinutes,genres,s1_ep_count,s1_mean_rating,s1_std_rating,s1_rating_slope,s1_vote_dropoff,pilot_votes
0,tt0429087,tvSeries,Shantaram,Shantaram,0,2022.0,2022.0,52,"Action,Adventure,Crime",12.0,7.700000,0.173205,-0.005594,0.608033,722.0
1,tt0475784,tvSeries,Westworld,Westworld,0,2016.0,2022.0,60,"Drama,Mystery,Sci-Fi",10.0,8.860000,0.438634,0.112727,1.340722,25129.0
2,tt0489974,tvSeries,Carnival Row,Carnival Row,0,2019.0,2023.0,56,"Crime,Drama,Fantasy",8.0,7.825000,0.326917,0.114286,0.809496,3454.0
3,tt0804484,tvSeries,Foundation,Foundation,0,2021.0,NaN,60,"Drama,Sci-Fi",10.0,7.490000,0.403609,0.049091,0.543780,7709.0
4,tt0808224,tvSeries,The Death of Bunny Munro,The Death of Bunny Munro,0,2025.0,2025.0,85,Drama,6.0,7.333333,0.418994,0.120000,0.685185,162.0
5,tt0808491,tvSeries,The Swarm,The Swarm,0,2023.0,2023.0,46,"Drama,Mystery,Sci-Fi",8.0,6.325000,0.243670,-0.069048,0.654378,651.0
6,tt10005938,tvSeries,Thanks a Million,Thanks a Million,0,2020.0,NaN,NaN,"Reality-TV,Short",3.0,9.000000,0.588784,-0.700000,0.059829,117.0
7,tt10009170,tvSeries,Blood of Zeus,Blood of Zeus,0,2020.0,2025.0,30,"Action,Adventure,Animation",8.0,7.987500,0.337037,0.091667,0.892601,1257.0
8,tt10011298,tvSeries,Dragon's Dogma,Dragon's Dogma,0,2020.0,2020.0,NaN,"Action,Adventure,Animation",7.0,6.842857,0.191663,0.053571,0.839552,268.0
9,tt10011518,tvSeries,The Darkness,The Darkness,0,2024.0,NaN,NaN,Mystery,6.0,7.000000,0.115470,0.022857,0.730435,115.0


In [60]:
# Upload TMBD dataset to get info about the status
filename = path + "TMDB_tv_dataset_v3.csv" 
tmdb_full = pd.read_csv(filename)

In [61]:
tmdb_full.dtypes

id                        int64
name                        str
number_of_seasons         int64
number_of_episodes        int64
original_language           str
vote_count                int64
vote_average            float64
overview                    str
adult                      bool
backdrop_path               str
first_air_date              str
last_air_date               str
homepage                    str
in_production              bool
original_name               str
popularity              float64
poster_path                 str
type                        str
status                      str
tagline                     str
genres                      str
created_by                  str
languages                   str
networks                    str
origin_country              str
spoken_languages            str
production_companies        str
production_countries        str
episode_run_time          int64
dtype: object

We have the variable status here that we will need, but columns are way too many which might result in a heavy dataset and slow procedures.

In [62]:
tmdb = tmdb_full[["name","first_air_date","status","number_of_seasons","networks","popularity"]].copy()

In [63]:
tmdb.head()

,name,first_air_date,status,number_of_seasons,networks,popularity
0,Game of Thrones,2011-04-17,Ended,8,HBO,1083.917
1,Money Heist,2017-05-02,Ended,3,"Netflix, Antena 3",96.354
2,Stranger Things,2016-07-15,Returning Series,4,Netflix,185.711
3,The Walking Dead,2010-10-31,Ended,11,AMC,489.746
4,Lucifer,2016-01-25,Ended,6,"FOX, Netflix",416.668


In [79]:
tmdb["startYear"] = pd.to_datetime(tmdb["first_air_date"], errors = "coerce").dt.year

In [80]:
tmdb.head()

,name,first_air_date,status,number_of_seasons,networks,popularity,startYear
0,Game of Thrones,2011-04-17,Ended,8,HBO,1083.917,2011.0
1,Money Heist,2017-05-02,Ended,3,"Netflix, Antena 3",96.354,2017.0
2,Stranger Things,2016-07-15,Returning Series,4,Netflix,185.711,2016.0
3,The Walking Dead,2010-10-31,Ended,11,AMC,489.746,2010.0
4,Lucifer,2016-01-25,Ended,6,"FOX, Netflix",416.668,2016.0


In [64]:
tmdb["status"].value_counts()

status
Ended               93232
Returning Series    67507
Canceled             4634
In Production        2380
Planned               650
Pilot                 236
Name: count, dtype: int64

We have some different status, we will focus only on <b>canceled at the first season</b> and tv shows with two seasons or more.

In [81]:
tmdb_2 = tmdb[((tmdb["status"] == "Canceled") & (tmdb["number_of_seasons"] == 1)) | (tmdb["number_of_seasons"] > 1)]

To have a better modeling, we create now our target variable: <b>is_canceled</b> set to 1 if the show has been canceled after one season, 0 otherwise (finished shows, in production, canceled after more seasons etc). 

In [82]:
# Create the target variable: isCanceled
tmdb_2["is_canceled"] = np.where((tmdb_2["status"] == "Canceled") & (tmdb_2["number_of_seasons"] == 1), 1, 0)

In [83]:
tmdb_2["is_canceled"].value_counts(normalize = True)*100

is_canceled
0    92.020242
1     7.979758
Name: proportion, dtype: float64

We have an imbalance: **92%** of the shows are not canceled (as expected), so we need to keep this in mind for modeling.

Now we need to merge the datasets, but they don't have the same IDs, so we can only use the show's title. Before merging, we need to clean the titles by removing spaces and converting everything to lowercase.

In [84]:
tmdb_2["clean_title"] = (tmdb_2["name"].str.lower()).str.strip()
imdb_df["clean_title"] = (imdb_df["primaryTitle"].str.lower()).str.strip()

In [85]:
tmdb_2.head()

,name,first_air_date,status,number_of_seasons,networks,popularity,startYear,is_canceled,clean_title
0,Game of Thrones,2011-04-17,Ended,8,HBO,1083.917,2011.0,0,game of thrones
1,Money Heist,2017-05-02,Ended,3,"Netflix, Antena 3",96.354,2017.0,0,money heist
2,Stranger Things,2016-07-15,Returning Series,4,Netflix,185.711,2016.0,0,stranger things
3,The Walking Dead,2010-10-31,Ended,11,AMC,489.746,2010.0,0,the walking dead
4,Lucifer,2016-01-25,Ended,6,"FOX, Netflix",416.668,2016.0,0,lucifer


In [86]:
df = imdb_df.merge(tmdb_2, on = ["clean_title","startYear"], how = "inner")
df.head(10)

,tconst,titleType,primaryTitle,originalTitle,isAdult,startYear,endYear,runtimeMinutes,genres,s1_ep_count,...,s1_vote_dropoff,pilot_votes,clean_title,name,first_air_date,status,number_of_seasons,networks,popularity,is_canceled
0,tt0429087,tvSeries,Shantaram,Shantaram,0,2022.0,2022.0,52,"Action,Adventure,Crime",12.0,...,0.608033,722.0,shantaram,Shantaram,2022-10-14,Canceled,1,Apple TV+,19.813,1
1,tt0475784,tvSeries,Westworld,Westworld,0,2016.0,2022.0,60,"Drama,Mystery,Sci-Fi",10.0,...,1.340722,25129.0,westworld,Westworld,2016-10-02,Canceled,4,HBO,66.988,0
2,tt0489974,tvSeries,Carnival Row,Carnival Row,0,2019.0,2023.0,56,"Crime,Drama,Fantasy",8.0,...,0.809496,3454.0,carnival row,Carnival Row,2019-08-29,Ended,2,Prime Video,34.433,0
3,tt0804484,tvSeries,Foundation,Foundation,0,2021.0,NaN,60,"Drama,Sci-Fi",10.0,...,0.543780,7709.0,foundation,Foundation,2021-09-24,Returning Series,2,Apple TV+,142.666,0
4,tt10005938,tvSeries,Thanks a Million,Thanks a Million,0,2020.0,NaN,NaN,"Reality-TV,Short",3.0,...,0.059829,117.0,thanks a million,Thanks a Million,2020-04-06,Returning Series,2,"Quibi, The Roku Channel",2.189,0
5,tt10016814,tvSeries,Unidentified: Inside America's UFO Investigation,Unidentified: Inside America's UFO Investigation,0,2019.0,2020.0,NaN,"Documentary,History,Sci-Fi",6.0,...,0.701299,77.0,unidentified: inside america's ufo investigation,Unidentified: Inside America's UFO Investigation,2019-05-31,Returning Series,2,History,3.238,0
6,tt10023178,tvSeries,A Little Late with Lilly Singh,A Little Late with Lilly Singh,0,2019.0,2021.0,30,"Comedy,Talk-Show",97.0,...,0.183857,223.0,a little late with lilly singh,A Little Late with Lilly Singh,2019-09-16,Canceled,2,NBC,514.265,0
7,tt10023294,tvSeries,That's My Jam,That's My Jam,0,2021.0,NaN,NaN,"Game-Show,Music",7.0,...,0.164706,85.0,that's my jam,That's My Jam,2021-11-29,Returning Series,2,NBC,8.021,0
8,tt10036104,tvSeries,Cursed Films,Cursed Films,0,2020.0,2022.0,30,"Documentary,Horror",5.0,...,0.908012,337.0,cursed films,Cursed Films,2020-04-02,Returning Series,2,Shudder,6.453,0
9,tt10037034,tvSeries,Gentefied,Gentefied,0,2020.0,2021.0,NaN,Comedy,10.0,...,0.692982,114.0,gentefied,Gentefied,2020-02-21,Canceled,2,Netflix,5.034,0


In [89]:
df.drop(columns=["clean_title"] , inplace = True) # remove the clean name, no need anymore

The **get_release_strategy** function queries the TVMaze API to retrieve exact Season 1 air dates.
This allows us to create the **is_binge** feature, distinguishing between all-at-once (binge) and weekly release models.


In [95]:
import requests
import time
import numpy as np

def get_release_strategy(imdb_id):
    try:
        # 1. Get TVMaze ID using the IMDb ID (tconst)
        url_lookup = f"https://api.tvmaze.com/lookup/shows?imdb={imdb_id}"
        r1 = requests.get(url_lookup, timeout=10)
        
        # Handle Rate Limit (Too Many Requests)
        if r1.status_code == 429:
            time.sleep(5)
            return get_release_strategy(imdb_id)
        if r1.status_code != 200:
            return pd.Series({'s1_release_span_days': np.nan, 'is_binge': np.nan})
            
        tvmaze_id = r1.json()['id']
        
        # 2. Fetch all episodes to extract air dates for Season 1
        url_eps = f"https://api.tvmaze.com/shows/{tvmaze_id}/episodes"
        r2 = requests.get(url_eps, timeout=10)
        eps = r2.json()
        
        # Filter dates for Season 1 only
        s1_dates = [pd.to_datetime(e['airdate']) for e in eps if e.get('season') == 1 and e.get('airdate')]
        
        if len(s1_dates) < 2:
            return pd.Series({'s1_release_span_days': np.nan, 'is_binge': np.nan})
            
        # Calculate days between first and last episode
        span_days = (max(s1_dates) - min(s1_dates)).days
        
        # If all episodes aired within 2 days, it's a binge release model
        is_binge = 1 if span_days <= 2 else 0
        
        return pd.Series({'s1_release_span_days': span_days, 'is_binge': is_binge})
        
    except Exception:
        return pd.Series({'s1_release_span_days': np.nan, 'is_binge': np.nan})

# Apply the function with a progress indicator
print("Fetching release dates from TVMaze API... (this might take a few minutes)")

# We use a loop to respect the 20 requests / 10 seconds limit
results = []
for i, imdb_id in enumerate(df['tconst']):
    results.append(get_release_strategy(imdb_id))
    # Print progress every 100 series
    if (i + 1) % 100 == 0:
        print(f"Processed {i + 1} / {len(df)} series...")
        
    time.sleep(0.5)  # Delay to respect rate limits

Fetching release dates from TVMaze API... (this might take a few minutes)
Processed 100 / 2143 series...
Processed 200 / 2143 series...
Processed 300 / 2143 series...
Processed 400 / 2143 series...
Processed 500 / 2143 series...
Processed 600 / 2143 series...
Processed 700 / 2143 series...
Processed 800 / 2143 series...
Processed 900 / 2143 series...
Processed 1000 / 2143 series...
Processed 1100 / 2143 series...
Processed 1200 / 2143 series...
Processed 1300 / 2143 series...
Processed 1400 / 2143 series...
Processed 1500 / 2143 series...
Processed 1600 / 2143 series...
Processed 1700 / 2143 series...
Processed 1800 / 2143 series...
Processed 1900 / 2143 series...
Processed 2000 / 2143 series...
Processed 2100 / 2143 series...


In [99]:
api_features = pd.DataFrame(results, index=df.index)

In [100]:
api_features.head()

,s1_release_span_days,is_binge
0,63.0,0.0
1,63.0,0.0
2,0.0,1.0
3,56.0,0.0
4,9.0,0.0


In [102]:
df = pd.concat([df, api_features], axis=1)

In [103]:
df.head()

,tconst,titleType,primaryTitle,originalTitle,isAdult,startYear,endYear,runtimeMinutes,genres,s1_ep_count,...,first_air_date,status,number_of_seasons,networks,popularity,is_canceled,s1_release_span_days,is_binge,s1_release_span_days,is_binge
0,tt0429087,tvSeries,Shantaram,Shantaram,0,2022.0,2022.0,52,"Action,Adventure,Crime",12.0,...,2022-10-14,Canceled,1,Apple TV+,19.813,1,63.0,0.0,63.0,0.0
1,tt0475784,tvSeries,Westworld,Westworld,0,2016.0,2022.0,60,"Drama,Mystery,Sci-Fi",10.0,...,2016-10-02,Canceled,4,HBO,66.988,0,63.0,0.0,63.0,0.0
2,tt0489974,tvSeries,Carnival Row,Carnival Row,0,2019.0,2023.0,56,"Crime,Drama,Fantasy",8.0,...,2019-08-29,Ended,2,Prime Video,34.433,0,0.0,1.0,0.0,1.0
3,tt0804484,tvSeries,Foundation,Foundation,0,2021.0,NaN,60,"Drama,Sci-Fi",10.0,...,2021-09-24,Returning Series,2,Apple TV+,142.666,0,56.0,0.0,56.0,0.0
4,tt10005938,tvSeries,Thanks a Million,Thanks a Million,0,2020.0,NaN,NaN,"Reality-TV,Short",3.0,...,2020-04-06,Returning Series,2,"Quibi, The Roku Channel",2.189,0,9.0,0.0,9.0,0.0


In [106]:
# Storing the final dataset.

filename = path + 'tv_series_final_dataset.csv'
df.to_csv(filename, index=False)